In [ ]:
"""
import langchain

print(langchain.__version__)

for langchain version 1.2.10
"""

from dotenv import load_dotenv
import os

load_dotenv()
api_key=os.getenv("")


# Set HuggingFace cache directory
os.environ["HF_HOME"] = "/Users/kristalshrestha/Documents/Code/LLM_Scratch/models"

# 1.Models

In [ ]:
# 1.1 LLM
from langchain_google_genai import GoogleGenerativeAI
from langchain_huggingface import (
    HuggingFaceEndpoint,
    ChatHuggingFace,
    HuggingFacePipeline,
)

In [ ]:
# 1.2 ChatModels
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# 1.3 Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

# 2.Prompts

In [ ]:
# 2.Prompts
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import load_prompt

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # not recommended,fails to dynamically fill variables ,so use ChatPrompt Template

from langchain_core.prompts import ChatPromptTemplate
# make tuples to make it work
# this also work (recommended)
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful {domain} expert"),
        ("human", "Explain in simple terms,what is {topic}"),
    ]
)
prompt = chat_template.invoke({"domain": "cricket", "topic": "Dusra"})


### 3.Structured OUTPUTS

In [ ]:
from typing import Annotated, Optional, Literal
from pydantic import BaseModel, Field


### 4.OUTPUT PARSER

In [ ]:
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser,PydanticOutputParser

### 5.CHAINS/Runnables

In [ ]:
from langchain_core.runnables import RunnableParallel,RunnableSequence,RunnablePassthrough,RunnableLambda,RunnableBranch

### 6.RAG

##### 6.1 Document Loader


In [ ]:
from langchain_community.document_loaders import TextLoader,PyPDFLoader,WebBaseLoader,CSVLoader

#### 6.2 Document Splitters

In [1]:
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker


#### 6.3 Vector Database

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

vector_store=Chroma(
    embedding_function=hf_embeddings_api,
    persist_directory="./self_practice_chroma_db",#where to store on disk,
    collection_name="sample" #collection name ~ similar to table in RDBMS
)
# add documents to the vector store
added_ids=vector_store.add_documents(docs)

#view the IDs assigned to each document
print("Document IDs:",added_ids)


### Direct all those 3

vectorstore=Chroma.from_documents(
    documents=documents,
    embedding=hf_embeddings_api,
    persist_directory="./2VectorDatabaseChroma",
    collection_name="table1"
)

#### 6.4 Retrievers

In [ ]:
retriever=vectorstore.as_retriever(search_kwargs={
    "k":2
})

## Agentic AI

## TOOLS IN LANGCHAIN

## Prebuilt Tool

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

#Create an instance of the tool
search_tool=DuckDuckGoSearchRun()


# Invoke the tool with a search query
results=search_tool.invoke("top news in nepal today")

# Print the result
print(results)

In [ ]:
from langchain_community.tools import ShellTool

shell_tool=ShellTool()
# Invoke the tool with a shell command
results=shell_tool.invoke("whoami") # or use ls command

print(results)
#output :(Note: This will show the username of the machine where the code is running.)

## Custom tool

In [ ]:
from langchain_core.tools import tool

# Step 1, 2, 3, and 4: Define the function and decorate it with @tool
@tool
def multiply(a:int,b:int)->int:
    """Multiplies two numbers"""
    return a * b


# The function `multiply` is now a tool.
# It has attributes like .name, .description, and .args
# Check its attributes
print(f"Tool Name: {multiply.name}")
print(f"Tool Description: {multiply.description}")
print(f"Tool Arguments: {multiply.args}")


# You can use the tool directly by calling its .invoke method and provide dictionary with inputs required to that function
result=multiply.invoke({"a":3,"b":5})
print(result)

## strict by using pydantic

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel,Field
from typing import Annotated,Literal

# 1. Define the Pydantic model for the input
class MultiplyInput(BaseModel):
    a:Annotated[int,Field(...,description="First number")]
    b:Annotated[int,Field(...,description="second number")]

# 2. Define the function logic
def multiply(a:int,b:int)->int:
    """Multiply two numbers"""
    return a*b

#3.Create a StucturedTool
multiply_tool=StructuredTool(
    func=multiply, # above defined function
    name="multiply_tool",
    description="Multiply two numbers",
    args_schema=MultiplyInput, #This enforces the schema
)


#Use the tool
result=multiply_tool.invoke(
    {
        "a":2,
        "b":3
    }
)
print(result) #Output:6

### Best way using Baseclass Tool

In [ ]:
from langchain_core.tools import BaseTool
from typing import Type,Annotated
from pydantic import BaseModel,Field

# Define a input schema (Pydantic Model)
class MultiplyInput(BaseModel):
    a:Annotated[int,Field(...,description="The first number to add")]
    b:Annotated[int,Field(...,description="The second number to add")]

# 2. Create a class that inherits from BaseTool
class MultiplyTool(BaseTool):
    name:str="multiply"
    description:str="Multiply two numbers"
    args_schema:Type[BaseModel]=MultiplyInput #This enforces the schema using pydantic model

    #3 Implement the _run method
    def _run(self,a:int,b:int)->int:
        """Synchronous method to multiply two numbers"""
        return a*b
    
    # Optional: Implement an async version
    # async def _arun(self, a: int, b: int) -> int:
    #     """Async version."""
    #     return a * b


# Use the Tool
my_tool=MultiplyTool()
result=my_tool.invoke({"a":2,"b":3})
print(result) #Ouput:6

## Toolkits

In [ ]:

# 1st tool  for Multiply
from langchain_core.tools import BaseTool
from typing import Type,Annotated
from pydantic import BaseModel,Field

# Define a input schema (Pydantic Model)
class MultiplyInput(BaseModel):
    a:Annotated[int,Field(...,description="The first number to multiply")]
    b:Annotated[int,Field(...,description="The second number to multiply")]

# 2. Create a class that inherits from BaseTool
class MultiplyTool(BaseTool):
    name:str="multiply"
    description:str="Multiply two numbers"
    args_schema:Type[BaseModel]=MultiplyInput #This enforces the schema using pydantic model

    #3 Implement the _run method
    def _run(self,a:int,b:int)->int:
        """Synchronous method to multiply two numbers"""
        return a*b
    
    # Optional: Implement an async version
    # async def _arun(self, a: int, b: int) -> int:
    #     """Async version."""
    #     return a * b


# 2nd tool  for add



# Define a input schema (Pydantic Model)
class AddInput(BaseModel):
    a:Annotated[int,Field(...,description="The first number to add")]
    b:Annotated[int,Field(...,description="The second number to add")]

# 2. Create a class that inherits from BaseTool
class AddTool(BaseTool):
    name:str="add"
    description:str="add two numbers"
    args_schema:Type[BaseModel]=AddInput #This enforces the schema using pydantic model defined above

    #3 Implement the _run method
    def _run(self,a:int,b:int)->int:
        """Synchronous method to add two numbers"""
        return a+b
    
    # Optional: Implement an async version
    # async def _arun(self, a: int, b: int) -> int:
    #     """Async version."""
    #     return a + b


class MathToolkit():
    def get_tools(self):
        """Return the list of tools in this toolkit"""
        return [MultiplyTool(),AddTool()]

#Use the toolkit
math_tools=MathToolkit()
all_math_tools=math_tools.get_tools()

# You can now use all tools in the toolkit
for tool in all_math_tools:
    print(f"Tool Name: {tool.name}")
    print(f"Tool Description: {tool.description}\n")

### TOOL CALLING

-  bind tool to an LLM

In [ ]:
#---------- Import necessary libraries----------------
import requests
import json
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated

import os
from dotenv import load_dotenv

load_dotenv()
exchange_rate_api_key=os.getenv("exchange_rate_api")
model_api_key=os.getenv("GOOGLE_API_KEY")

#------1. Create Tools -------------------
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """Fetches the real-time currency conversion factor between a base currency and a target currency."""
    url = f"https://v6.exchangerate-api.com/v6/{exchange_rate_api_key}/pair/{base_currency}/{target_currency}"

    response=requests.get(url)
    return response.json() ## we get conversion rate from here

@tool
def convert(base_currency_value: float, conversion_rate:Annotated[float,InjectedToolArg]) -> float:
    """Given a base currency value and a conversion rate , calculates the target currency value based on base currency value."""

    return base_currency_value*conversion_rate



# --- 2. Bind Tools to LLM --- Tool Binding
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=model_api_key)
tools=[get_conversion_factor,convert]
model_with_tools=model.bind_tools(tools)

# --- 3. The Full Agentic-Like Flow ---
messages = [HumanMessage(content="What is the conversion factor between USD and INR? Based on that, convert 10 USD to INR.")]

# First LLM call: It decides to use both tools.
ai_response = model_with_tools.invoke(messages)
messages.append(ai_response)


# Process tool calls
for tool_call in ai_response.tool_calls:
    if tool_call["name"] == "get_conversion_factor":
        # Execute the first tool
        tool_result1 = get_conversion_factor.invoke(tool_call)
        # Store the conversion rate for later injection by fetching it from the tool_result
        conversion_rate=json.loads(tool_result1.content)["conversion_rate"] #changing to dictionary to access conversion_rate key
        print(f"Conversion rate fetched from api: {conversion_rate}")
        messages.append(tool_result1)
        
    if tool_call["name"]=="convert":
        # For the second tool, get its arguments
        # Inject the conversion_rate we got from the first tool
        tool_call["args"]["conversion_rate"]=conversion_rate
        # Execute the tool with the injected argument
        tool_result2 = convert.invoke(tool_call)
        messages.append(tool_result2)

# Final LLM call: It has the full history and can generate a final answer.
final_response = model_with_tools.invoke(messages)
print(final_response.content)